# 🔀 Notebook Bônus — Multi-Chains e Multi-Modelos com LCEL

**Disciplina:** Prompt Engineering and Artificial Intelligence
**Instituição:** FIAP — Ciência da Computação · 2026
**Professor:** Jorge Luiz Gomes
**Material bônus — não vinculado a nenhuma aula do plano do 2º semestre**
**⏱️ ~30min de leitura + execução**
**🐍 RunnableParallel · RunnableLambda · Composição sequencial**

---

## 🎯 Objetivo

Mostrar 3 formas de compor múltiplas chains LCEL chamando **mais de um modelo** na mesma pipeline: sequencial (uma chain alimenta a próxima), paralela (fan-out para comparar respostas) e condicional (roteia para o modelo certo conforme o input). Nenhuma aula do plano cobre os três padrões juntos — este notebook é material extra, sem lacunas e sem entrega associada.

---

## Como usar este notebook

- Rode as células na ordem, de cima para baixo (`Shift+Enter`).
- Não há lacunas — é demonstrativo, para rodar e observar o comportamento de cada padrão.
- Pré-requisito: LCEL básico (`prompt | llm | parser`) — visto na Aula 01.

## ⚙️ Setup — dois modelos, uma API key

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Dois modelos da mesma família, dois tamanhos — a base de todo o notebook
llm_leve   = ChatOllama(model="gpt-oss:20b",  temperature=0.7)   # mais rápido, mais barato
llm_grande = ChatOllama(model="gpt-oss:120b", temperature=0.7)   # mais lento, respostas mais completas

## 1️⃣ Sequencial — cada etapa usa um modelo diferente

A chain 1 roda no modelo leve e gera um rascunho rápido. A chain 2 roda no modelo grande e refina esse rascunho. A composição `{"rascunho": chain_rascunho} | chain_refino` passa o mesmo input pra `chain_rascunho` e usa a saída dela como entrada de `chain_refino` — sem código imperativo de "chamar A, guardar resultado, chamar B".

In [ ]:
prompt_rascunho = ChatPromptTemplate.from_messages([
    ("system", "Você é um redator técnico. Gere um rascunho curto, direto, sem formatação."),
    ("human",  "Tema: {tema}"),
])
chain_rascunho = prompt_rascunho | llm_leve | StrOutputParser()

prompt_refino = ChatPromptTemplate.from_messages([
    ("system", "Você é um editor sênior. Expanda o rascunho abaixo em um parágrafo completo, com um exemplo prático."),
    ("human",  "Rascunho: {rascunho}"),
])
chain_refino = prompt_refino | llm_grande | StrOutputParser()

# Composição sequencial: a saída da chain 1 vira a entrada da chain 2
chain_sequencial = {"rascunho": chain_rascunho} | chain_refino

resultado = chain_sequencial.invoke({"tema": "computação quântica para iniciantes"})
print(resultado)

## 2️⃣ Paralelo (fan-out) — o mesmo input em dois modelos ao mesmo tempo

`RunnableParallel` roda `chain_leve` e `chain_grande` **concorrentemente**, não uma depois da outra — o tempo total fica perto do maior dos dois tempos individuais, não da soma. Útil para comparar qualidade vs. custo/latência entre modelos antes de decidir qual usar em produção.

In [ ]:
import time

prompt_comparativo = ChatPromptTemplate.from_messages([
    ("system", "Responda de forma direta em até 3 frases."),
    ("human",  "{pergunta}"),
])

chain_leve_cmp   = prompt_comparativo | llm_leve   | StrOutputParser()
chain_grande_cmp = prompt_comparativo | llm_grande | StrOutputParser()

comparador = RunnableParallel(leve=chain_leve_cmp, grande=chain_grande_cmp)

inicio = time.perf_counter()
respostas = comparador.invoke({"pergunta": "O que é overfitting? Responda em uma frase."})
duracao = time.perf_counter() - inicio

print(f"gpt-oss:20b  → {respostas['leve']}")
print(f"gpt-oss:120b → {respostas['grande']}")
print(f"\nTempo total: {duracao:.1f}s — os dois modelos rodaram ao mesmo tempo, não em sequência")

## 3️⃣ Condicional (routing) — classifica o input e escolhe o modelo

Extensão direta do Router Chain (Aula 12): lá o router escolhe **qual chain** rodar; aqui o router escolhe **qual modelo** rodar. O próprio modelo leve funciona como classificador — pergunta simples vai pro modelo leve, pergunta complexa vai pro modelo grande. Economiza custo e latência sem perder qualidade nas perguntas que realmente precisam do modelo grande.

In [ ]:
prompt_classificador = ChatPromptTemplate.from_messages([
    ("system", "Classifique a pergunta como SIMPLES (resposta factual curta) ou COMPLEXA (exige raciocínio ou múltiplos passos). Responda só com uma palavra: SIMPLES ou COMPLEXA."),
    ("human",  "{pergunta}"),
])
classificador = prompt_classificador | llm_leve | StrOutputParser()

prompt_resposta = ChatPromptTemplate.from_messages([
    ("system", "Responda de forma clara e completa."),
    ("human",  "{pergunta}"),
])
resposta_leve   = prompt_resposta | llm_leve   | StrOutputParser()
resposta_grande = prompt_resposta | llm_grande | StrOutputParser()

def rotear_por_complexidade(input_dict):
    pergunta = input_dict["pergunta"]
    classificacao = classificador.invoke({"pergunta": pergunta}).strip().upper()
    usar_leve = "SIMPLES" in classificacao
    print(f"[roteador] classificou como {classificacao} → usando {'gpt-oss:20b' if usar_leve else 'gpt-oss:120b'}")
    chain_escolhida = resposta_leve if usar_leve else resposta_grande
    return chain_escolhida.invoke({"pergunta": pergunta})

roteador = RunnableLambda(rotear_por_complexidade)

print(roteador.invoke({"pergunta": "Qual é a capital da França?"}))
print()
print(roteador.invoke({"pergunta": "Compare RAG ingênuo e RAG avançado, com os trade-offs de cada um."}))

## 📊 Quando usar cada padrão

| Padrão | Quando usar | Trade-off |
|---|---|---|
| Sequencial | Etapas especializadas (rascunho → refino, extração → formatação) | Latência soma (t1 + t2) |
| Paralelo | Comparar/ensemble de respostas para o mesmo input | Custo soma; latência é o máximo dos dois |
| Condicional | Otimizar custo/latência roteando por complexidade | Depende de um classificador confiável |

## 📚 Referências

- Docs LangChain — LCEL: composição declarativa de chains com o operador \|. python.langchain.com/docs/concepts/lcel
- Docs LangChain — Runnables: RunnableParallel (execução concorrente) e RunnableLambda (lógica Python arbitrária como Runnable). python.langchain.com/docs/concepts/runnables
- Docs langchain-ollama — ChatOllama: múltiplas instâncias, um modelo por instância, mesma interface. python.langchain.com/docs/integrations/chat/ollama

---

**Conexão com o plano do semestre:** este notebook estende o Router Chain da Aula 12 (roteamento entre chains) e prepara o terreno para o StateGraph da Aula 13, onde nodes podem chamar modelos diferentes conforme o estado do grafo.

---

*Material bônus — Prof. Jorge Luiz Gomes · FIAP · Prompt Engineering and Artificial Intelligence · 2026*